# 🧮 Teach a Vision Model to Read Math — GRPO + OpenEnv

Reinforcement-learning a **vision-language model** to turn images of math formulas into **LaTeX**, using TRL's
`GRPOTrainer` and an **OpenEnv** environment that supplies *both* the image and the reward.

**What you'll do:**
1. Install + connect to the LaTeX-OCR environment (hosted, or run it locally)
2. **Evaluate the base model** on held-out test images (a starting score)
3. **Train** with GRPO (live reward curve on a Trackio dashboard)
4. **Evaluate again** and see the improvement (the delta)
5. **Push** your model to the Hub

> 💡 A GPU is required. An **A100** (Colab Pro / a beefy runtime) is recommended for a 4B model; drop to a 2B model or
> fewer generations on smaller GPUs.

## 1. Install

⏳ Takes ~2–3 min and prints nothing (output is hidden). The `[*]` turns into a number when it's done — then move on.

In [ ]:
%%capture
# trl (from source: VLM + environment_factory) + the LaTeX-OCR OpenEnv package + LoRA/util deps
!pip install -q "trl @ git+https://github.com/huggingface/trl.git" \
    "openenv-latex_ocr_env @ git+https://github.com/adithya-s-k/OpenEnv.git@add-latex-ocr-multimodal-streaming-env#subdirectory=envs/latex_ocr_env" \
    peft trackio ipywidgets fastmcp websockets jmespath hf_transfer

## 🔑 Log in to Hugging Face

You need a free HF account and a **token with _write_ access** — create one at
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens). The token lets the notebook create
your Trackio dashboard Space and push the trained model to the Hub. Run the cell and paste it in.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # paste a WRITE token   (or instead: import os; os.environ["HF_TOKEN"] = "hf_...")

## 2. Settings

OpenEnv is a tiny server that **hands you a formula image** and **scores your LaTeX** (edit-distance + exact-match,
computed server-side). By default we use the free **hosted Space** — zero setup. Flip `USE_LOCAL_ENV = True` to run
the same environment on this machine instead (next cell).

In [ ]:
import os
from huggingface_hub import whoami
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

USERNAME           = whoami()["name"]        # from your login above — used to name your repos

MODEL              = "Qwen/Qwen3.5-4B"      # any image-text-to-text VLM (e.g. Qwen/Qwen3.5-2B for smaller GPUs)
USE_LOCAL_ENV      = False                   # False = hosted Space (no setup); True = serve locally (next cell)
ENV_URL            = "https://AdithyaSK-latex-ocr-env.hf.space"

N_EVAL_SAMPLES     = 50                       # test images to score for eval
NUM_TRAIN_SAMPLES  = 500                      # env tasks to train on
MAX_STEPS          = 30                        # optimization steps (controls how many tasks are actually consumed)
NUM_GENERATIONS    = 8

# Your repos (auto-named from your username). Set PUSH_REPO_ID = "" to skip pushing the model.
PUSH_REPO_ID       = f"{USERNAME}/qwen3.5-latex-ocr-grpo"   # trained model goes here

# Experiment tracking (Trackio). Turn OFF to train without any dashboard / Space.
USE_TRACKIO        = True
TRACKIO_SPACE_ID   = f"{USERNAME}/trackio-latex-ocr"        # dashboard Space (auto-created when USE_TRACKIO)
TRACKIO_PROJECT    = "latex-ocr-grpo"

print(f"👤 logged in as   : {USERNAME}")
print(f"🧠 model          : {MODEL}")
print(f"🌐 environment    : {'local (next cell)' if USE_LOCAL_ENV else ENV_URL}")
print(f"🏋️  train / steps  : {NUM_TRAIN_SAMPLES} tasks available · {MAX_STEPS} steps · {NUM_GENERATIONS} generations")
print(f"📏 eval samples   : {N_EVAL_SAMPLES}")
print(f"📈 tracking       : {'Trackio → ' + TRACKIO_SPACE_ID if USE_TRACKIO else 'off'}")
print(f"📤 push to        : {PUSH_REPO_ID or '(skipped)'}")

### (Optional) Run the environment locally

Only needed if `USE_LOCAL_ENV = True`. Starts the OpenEnv LaTeX-OCR server on this machine (CPU-only, runs
alongside your GPU training) and points `ENV_URL` at it.

Two knobs keep it fast:
- **`LATEX_OCR_MODE=materialize`** — loads + indexes the split so `reset(split, index)` is O(1) random access.
  (The other mode, `stream`, is a sequential cursor with *no* random access — our training/eval need indices, so
  streaming would be unusably slow here.)
- **`LATEX_OCR_MAX_ROWS`** — materialize would otherwise decode the *entire* split (68k+ train images). We cap it to
  just the rows we actually touch, so the server is ready in seconds instead of minutes.

In [ ]:
if USE_LOCAL_ENV:
    import subprocess, sys, time, requests
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)  # env server's UI import
    os.environ["LATEX_OCR_MODE"]        = "materialize"                    # random-access reset(split, index)
    os.environ["LATEX_OCR_MAX_ROWS"]    = str(max(NUM_TRAIN_SAMPLES, N_EVAL_SAMPLES))  # only load what we use
    os.environ["LATEX_OCR_MAX_SESSIONS"] = str(max(16, NUM_GENERATIONS))   # allow concurrent rollouts
    _server = subprocess.Popen(["python", "-c", "from latex_ocr_env.server.app import main; main()"])
    ENV_URL = "http://localhost:8000"
    for _ in range(90):
        try:
            if requests.get(f"{ENV_URL}/healthz", timeout=2).ok:
                print("✅ local env server ready:", ENV_URL); break
        except Exception:
            pass
        time.sleep(2)
    else:
        raise RuntimeError("local env server did not start — check the logs above")
else:
    print("Using hosted environment:", ENV_URL)

## 3. The environment wrapper + reward

`GRPOTrainer` talks to an environment through a small class: `reset()` returns the **image + instruction** (TRL puts
the image straight into the VLM's prompt), and the reward function submits the model's LaTeX with `step()` and reads
the **server-computed reward**.

**The dataset lives inside the environment.** It serves `train` and `test` splits of formula images; we never load a
dataset ourselves. Our local "dataset" is just a list of `{split, index}` rows telling the trainer which task to
`reset()` to — so every generation in a group sees the same image.

In [ ]:
import base64, re
from io import BytesIO
from PIL import Image
from datasets import Dataset
from latex_ocr_env import LatexOCRAction, LatexOCREnv

INSTRUCTION = "Transcribe the mathematical formula in the image to LaTeX. Output only the LaTeX code."

def _strip_fences(t):
    t = (t or "").strip()
    m = re.search(r"```(?:latex)?\s*(.*?)\s*```", t, flags=re.DOTALL)
    if m: t = m.group(1)
    return t.strip().strip("$").strip()

def _decode(b64, max_px=512):
    img = Image.open(BytesIO(base64.b64decode(b64))).convert("RGB")
    img.thumbnail((max_px, max_px), Image.LANCZOS)
    return img

class LatexOCREnv_GRPO:
    """environment_factory: the env supplies the image (reset) and the reward (step)."""
    def __init__(self):
        self.client = LatexOCREnv(base_url=ENV_URL, connect_timeout_s=60, message_timeout_s=180)
    def reset(self, split="train", index=None, **kw):
        idx = int(index) if index is not None else None
        obs = self.client.reset(split=split, index=idx).observation
        return [{"type": "image", "image": _decode(obs.image_base64)},
                {"type": "text", "text": obs.prompt or INSTRUCTION}]

def env_reward(completions, environments, **kw):
    return [float(e.client.step(LatexOCRAction(latex=_strip_fences(c[0]["content"]))).reward or 0.0)
            for e, c in zip(environments, completions)]

def build_dataset(n, split="train"):
    probe = LatexOCREnv(base_url=ENV_URL, connect_timeout_s=60)
    k = min(n, probe.num_tasks(split))
    print(f"{split}: using {k} of {probe.num_tasks(split)} tasks")
    return Dataset.from_dict({"split": [split] * k, "index": list(range(k))})

# --- peek: connect to the env and show one task so you can see what the model sees ---
from IPython.display import display
_probe = LatexOCREnv(base_url=ENV_URL, connect_timeout_s=60)
print(f"✅ connected to env  ·  train tasks: {_probe.num_tasks('train')}  ·  test tasks: {_probe.num_tasks('test')}")
_obs = _probe.reset(split="train", index=0).observation
print(f"prompt: {(_obs.prompt or INSTRUCTION)}")
print("example image the model must transcribe:")
display(_decode(_obs.image_base64))

## 4. Evaluation

A simple, honest score: for `N` **test-split** images, let the model generate LaTeX and grade it with the **same
reward** used in training. The average is our metric. We'll run it before *and* after training.

In [ ]:
import torch
from tqdm.auto import tqdm
from IPython.display import display

def generate_latex(model, processor, image, prompt, max_new_tokens=256):
    msgs = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True, return_dict=True,
        return_tensors="pt", enable_thinking=False,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]

def evaluate(model, processor, n_samples, split="test", show=3):
    """Average env reward over the first N `split` images (greedy). Prints progress + a few examples."""
    client = LatexOCREnv(base_url=ENV_URL, connect_timeout_s=60, message_timeout_s=180)
    # Generation is ~10x faster with the KV cache. After training, the model still has gradient
    # checkpointing on (which forces use_cache=False) and is in train mode — so switch to a fast
    # eval state here and restore it afterwards, so before/after evals run at the same speed.
    was_training = model.training
    was_gc = getattr(model, "is_gradient_checkpointing", False)
    model.eval()
    if was_gc: model.gradient_checkpointing_disable()
    model.config.use_cache = True
    scores, examples = [], []
    try:
        pbar = tqdm(range(n_samples), desc=f"evaluating [{split}]")
        for i in pbar:
            obs = client.reset(split=split, index=i).observation
            img = _decode(obs.image_base64)
            pred = generate_latex(model, processor, img, obs.prompt or INSTRUCTION)
            r = float(client.step(LatexOCRAction(latex=_strip_fences(pred))).reward or 0.0)
            scores.append(r)
            pbar.set_postfix(avg_reward=f"{sum(scores) / len(scores):.3f}")   # live running average
            if i < show: examples.append((img, _strip_fences(pred), r))
    finally:
        model.config.use_cache = False
        if was_gc: model.gradient_checkpointing_enable()
        if was_training: model.train()
    mean = sum(scores) / len(scores)
    print(f"\n📊 mean reward over {n_samples} [{split}] samples = {mean:.4f}\n")
    for j, (img, pred, r) in enumerate(examples):
        print(f"— example {j + 1}  ·  reward {r:.3f}")
        display(img)
        print(f"   model output: {pred[:160]}\n")
    return mean

## 5. Build the trainer

We load the VLM with **LoRA** (adapters — light on memory) and hand `GRPOTrainer` the `environment_factory` +
`env_reward`. `enable_thinking=False` keeps Qwen from emitting a long `<think>` block so it outputs LaTeX directly.

We also turn on **[Trackio](https://huggingface.co/blog/trackio)** — a free, local-first experiment tracker with a
`wandb`-style API. Setting `report_to="trackio"` + `trackio_space_id` streams the reward/loss/KL curves live to a
Hugging Face Space (a real-time dashboard you can share). We'll embed it right in this notebook after training.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig

train_dataset = build_dataset(NUM_TRAIN_SAMPLES, "train")

peft_config = LoraConfig(task_type="CAUSAL_LM", r=16, lora_alpha=32, lora_dropout=0.05,
                         target_modules=["q_proj", "v_proj"])

trainer = GRPOTrainer(
    model=MODEL,
    train_dataset=train_dataset,
    reward_funcs=env_reward,
    environment_factory=LatexOCREnv_GRPO,
    peft_config=peft_config,
    args=GRPOConfig(
        output_dir="latex_ocr_grpo",
        model_init_kwargs={"dtype": "bfloat16"},
        num_generations=NUM_GENERATIONS,
        per_device_train_batch_size=NUM_GENERATIONS,
        max_steps=MAX_STEPS,
        max_completion_length=256,
        temperature=0.9,
        learning_rate=1e-5,
        chat_template_kwargs={"enable_thinking": False},
        mask_truncated_completions=True,
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=1, save_strategy="no",
        # --- Trackio logging (toggled by USE_TRACKIO) ---
        report_to=("trackio" if USE_TRACKIO else "none"),
        run_name="train",
        project=TRACKIO_PROJECT,
        trackio_space_id=(TRACKIO_SPACE_ID if USE_TRACKIO else None),
        trackio_static_space_id=False,   # keep the live Space (skip freezing it)
    ),
)
processor = trainer.processing_class

_trainable = trainer.model.num_parameters(only_trainable=True) / 1e6
print(f"✅ trainer ready  ·  {MODEL}")
print(f"   trainable params: {_trainable:.1f}M (LoRA)  ·  dataset: {len(train_dataset)} env tasks  ·  {MAX_STEPS} steps")
print(f"   metrics → Trackio project '{TRACKIO_PROJECT}' on {TRACKIO_SPACE_ID}" if USE_TRACKIO else "   tracking: off")

## 6. Baseline — how good is it *before* training?

(With LoRA freshly initialized, this is effectively the base model's score.)

In [ ]:
print("Scoring the untrained model on the test split — watch the examples below...")
baseline = evaluate(trainer.model, processor, N_EVAL_SAMPLES, split="test")

## 7. Train

Each step prints its metrics (loss, KL, and `rewards/env_reward/mean` — the number that should climb). The same
curves stream live to your Trackio dashboard, which we embed a couple of cells down.

In [ ]:
print("🚀 training — watch 'rewards/env_reward/mean' trend up over the steps below\n")
trainer.train()

## 8. Did it improve?

Same `N` test images, same reward — now with the trained adapter.

In [ ]:
trained = evaluate(trainer.model, processor, N_EVAL_SAMPLES, split="test")
print("=" * 48)
print(f"  before training : {baseline:.4f}")
print(f"  after training  : {trained:.4f}")
print(f"  improvement (Δ) : {trained - baseline:+.4f}")
print("=" * 48)

## 📈 Live dashboard (Trackio)

Training already streamed its reward curve to Trackio. Now we log the **eval** as its own run — `test_reward` at
step 0 (before) and at the final step (after) — so the before→after jump shows up as a line. Then we embed the
whole dashboard (train + eval) right here in the notebook.

In [ ]:
if USE_TRACKIO:
    import trackio
    # Log the baseline/after eval as a 2-point 'eval' run in the same project
    trackio.init(project=TRACKIO_PROJECT, name="eval", space_id=(TRACKIO_SPACE_ID or None))
    trackio.log({"test_reward": baseline}, step=0)
    trackio.log({"test_reward": trained}, step=MAX_STEPS)
    trackio.finish()

    # Embed the dashboard inline (train curves + eval line)
    from IPython.display import IFrame, display
    sub = TRACKIO_SPACE_ID.replace("/", "-").lower()
    display(IFrame(f"https://{sub}.hf.space/?project={TRACKIO_PROJECT}", width="100%", height=640))
else:
    print("Trackio is disabled (USE_TRACKIO = False) — skipping the dashboard.")

## 9. Push to the Hub

We **merge the LoRA adapter into the base weights** and push a standalone model (+ processor) — so anyone can load
it directly with `from_pretrained`, no adapter step needed. Set `PUSH_REPO_ID` in the settings cell first and log in
(`huggingface-cli login` or `notebook_login()`).

In [ ]:
if PUSH_REPO_ID:
    merged = trainer.model.merge_and_unload()   # fold LoRA into the base weights
    merged.push_to_hub(PUSH_REPO_ID)
    processor.push_to_hub(PUSH_REPO_ID)
    print("✅ pushed to https://huggingface.co/" + PUSH_REPO_ID)
else:
    print("Set PUSH_REPO_ID in the settings cell to push your model.")

## Recap

You evaluated a VLM, trained it with GRPO against an OpenEnv reward, measured the improvement, and pushed it.

**Try next:** more `MAX_STEPS`, a bigger `N_EVAL_SAMPLES` for a steadier score, a different `MODEL`, or
`USE_LOCAL_ENV=True` to run the environment yourself.